# 13 · Barrier & Autocallable Structures

Equity exotics combine path triggers, coupons, and barriers. The bindings expose JSON-like builders mirroring desk termsheets.

### Learning Objectives
- Configure knock-in/knock-out barrier options and price them with Monte Carlo GBM
- Build a multi-observation autocallable with step-down barriers and payoff caps

In [ ]:
from datetime import date

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data import MarketContext
from finstack.core.market_data.scalars import MarketScalar
from finstack.core.market_data.surfaces import VolSurface
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.valuations.instruments import BarrierOption, Autocallable
from finstack.valuations.pricer import create_standard_registry

val_date = date(2025, 1, 1)
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD.SOFR",
        val_date,
        [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (2.0, 0.90), (3.0, 0.86)],
    )
)
market.insert_surface(
    VolSurface(
        "SPX.VOL",
        expiries=[1.0, 2.0, 3.0],
        strikes=[4000.0, 4500.0, 5000.0, 5500.0],
        grid=[
            [0.20, 0.18, 0.16, 0.18],
            [0.22, 0.20, 0.18, 0.20],
            [0.24, 0.22, 0.20, 0.22],
        ],
    )
)
market.insert_price("SPX", MarketScalar.price(Money(5000.0, USD)))
market.insert_price("SPX.DIV", MarketScalar.unitless(0.015))
registry = create_standard_registry()


## 1. Down-and-In Barrier Call
Specify barrier level, barrier type, and underlying IDs. Pricing uses the Monte Carlo GBM engine.

In [ ]:
barrier_call = BarrierOption.builder(
    "SPX-BARRIER",
    "SPX",
    5000.0,
    4200.0,
    "call",
    "down_and_in",
    date(2026, 1, 2),
    Money(100000.0, USD),
    "USD.SOFR",
    "SPX",
    "SPX.VOL",
    div_yield_id="SPX.DIV",
    use_gobet_miri=False,
)
barrier_res = registry.price(barrier_call, "monte_carlo_gbm", market, as_of=val_date)
print("Barrier call PV:", barrier_res.value)


## 2. Autocallable Note
Autocallables combine observation calendars, knock-out barriers, coupon schedules, and final payoff rules. All parameters map directly to the builder.

In [ ]:
observation_dates = [
    date(2025, 4, 1),
    date(2025, 7, 1),
    date(2025, 10, 1),
    date(2026, 1, 1),
    date(2026, 4, 1),
    date(2026, 7, 1),
]
autocall = Autocallable.builder(
    "SPX-AUTO",
    "SPX",
    observation_dates,
    [1.00, 0.98, 0.96, 0.94, 0.92, 0.90],
    [0.02, 0.04, 0.06, 0.08, 0.10, 0.12],
    0.80,
    {"type": "participation", "rate": 1.0},
    1.0,
    1.5,
    Money(100000.0, USD),
    "USD.SOFR",
    "SPX",
    "SPX.VOL",
    div_yield_id="SPX.DIV",
)
auto_res = registry.price(autocall, "monte_carlo_gbm", market, as_of=val_date)
print("Autocallable PV:", auto_res.value)


## Summary
- Barrier options expose the same strike/barrier attributes desks use; Monte Carlo engines handle knock-in/out logic.
- Autocallables accept observation schedules, coupon ladders, and payoff metadata, letting you originate structured notes inside Python.